# 조합A + 5Day_Return — XGBoost

## 실험 목적

HF의 `full/index_price_dev.parquet`에서 `코스피 200`만 선택해 5Day_Return 추가 실험을 수행한다.
피처와 미래 5거래일 방향 라벨은 parquet 원시값으로 다시 계산한다.

## 공통 조건

| 항목 | 값 |
|---|---|
| 데이터 | HF 비공개 저장소의 고정 리비전 parquet |
| 홀드아웃 시작 | `20240901` — 이후 행 접근 금지 |
| 외부 검증 | expanding walk-forward 12폴드 |
| 최초 외부 학습 | 750거래일 |
| 외부 검증 | 폴드당 60거래일 |
| 학습·검증 gap | 5거래일 |
| 클래스 가중치 | 외부 폴드마다 내부 60일 검증으로 `None`/`balanced` 선택 |
| 가중치 선정 기준 | Accuracy·Macro F1·하락 Recall 조화평균 |
| 매매 평가 | 상승만 다음 시가 진입, 5슬리브·5거래일 보유, 왕복비용 0.05% |


In [1]:
# 어느 폴더에서 실행해도 프로젝트 모듈을 찾도록 루트를 확인합니다.
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from features.model_dataset import build_model_dataset  # noqa: E402
from models.experiment import evaluate_nested_class_weights  # noqa: E402
from models.notebook_experiment import summarize_notebook_experiment  # noqa: E402
from supply.hf_model_data import load_hf_index_prices  # noqa: E402

COMBINATION = "A"
RETURN_FEATURES = ('five_day_return',)
MODEL_NAME = "XGBoost"
CACHE_NAME = "A_five_day_XGBoost.json"

# MANIFEST와 parquet를 같은 HF 커밋에서 받고 SHA-256·홀드아웃 경계를 검사합니다.
snapshot = load_hf_index_prices()
dataset = build_model_dataset(
    snapshot.frame,
    COMBINATION,
    return_features=RETURN_FEATURES,
)
print("HF 리비전:", snapshot.repo_sha)
print("지수 parquet SHA-256:", snapshot.file_sha256)
print("학습 가능 기간:", dataset.frame["bas_dd"].min(), "~", dataset.frame["bas_dd"].max())
print("학습 가능 행:", len(dataset.frame))
print("피처:", list(dataset.feature_columns))


C:\Users\Administrator\Alpha_Stack\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HF 리비전: cf3759afebf73a59c8f6c9aa7265cccb56a38f27
지수 parquet SHA-256: 376c66434db688f42972d2011baee330641421f6b2ca32a8074a96db85ebc13a
학습 가능 기간: 20100201 ~ 20240822
학습 가능 행: 3592
피처: ['rsi_14', 'bb_bandwidth', 'hv_20', 'vol_ratio_20', 'five_day_return']


## 클래스 가중치 재튜닝

외부 검증 60일은 가중치 선정에 사용하지 않는다.

In [2]:
# 12개 외부 폴드 각각에서 과거 데이터만으로 가중치를 고르고 OOS를 평가합니다.
nested = evaluate_nested_class_weights(dataset, model_names=(MODEL_NAME,))
result = summarize_notebook_experiment(dataset, nested, MODEL_NAME)

fold_columns = [
    "fold",
    "selected_class_weight",
    "train_size",
    "train_end",
    "valid_start",
    "valid_end",
    "accuracy",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
    "delta_sharpe_net",
    "all_cash",
]
display(result.fold_results.loc[:, fold_columns].round(4))
display(result.weight_counts)


,fold,selected_class_weight,train_size,train_end,valid_start,valid_end,accuracy,macro_f1,down_recall,core_harmonic_mean,delta_sharpe_net,all_cash
0,1,NaN,750,20130201,20130212,20130508,0.3000,0.2782,0.2105,0.2569,-0.8522,False
1,2,balanced,1002,20140212,20140220,20140519,0.3500,0.3204,0.1538,0.2404,1.1700,False
2,3,balanced,1255,20150225,20150305,20150601,0.3333,0.2650,0.0952,0.1737,-0.9546,False
3,4,balanced,1507,20160303,20160311,20160608,0.3167,0.2799,0.0000,0.0000,2.5610,False
4,5,NaN,1760,20170310,20170320,20170616,0.5333,0.2642,0.0000,0.0000,-2.4453,False
5,6,balanced,2012,20180323,20180402,20180629,0.3333,0.3266,0.3846,0.3464,0.3257,False
6,7,balanced,2265,20190408,20190416,20190711,0.3167,0.3071,0.2381,0.2827,1.1244,False
7,8,balanced,2517,20200413,20200422,20200717,0.3833,0.3375,0.7500,0.4345,2.1711,False
8,9,balanced,2770,20210422,20210430,20210726,0.5500,0.5310,0.5833,0.5539,1.9030,False
9,10,NaN,3022,20220429,20220510,20220803,0.4333,0.3961,0.1579,0.2687,0.2359,False


,클래스 가중치,선택 폴드 수
0,None,3
1,balanced,9


## OOS 핵심지표와 백테스트 성과

In [3]:
# 전체 720개 OOS 예측을 합쳐 분류 성능과 혼동행렬을 계산합니다.
summary = result.summary
summary_table = pd.Series(
    {
        "전체 OOS 표본": summary["oos_rows"],
        "Accuracy": summary["accuracy"],
        "Macro F1": summary["macro_f1"],
        "하락 Recall": summary["down_recall"],
        "핵심지표 조화평균": summary["core_harmonic_mean"],
        "Balanced Accuracy": summary["balanced_accuracy"],
        "최빈 클래스 기준선": summary["majority_accuracy"],
        "ΔSharpe_net 폴드 중앙값": summary["delta_sharpe_net_median"],
        "전 기간 현금 폴드": summary["all_cash_folds"],
    },
    name="결과",
)
display(summary_table.to_frame().round(4))
display(result.confusion)
display(result.class_report.round(4))

# 뒤의 05.모델비교 노트북은 네 모델의 이 실행 결과만 읽습니다.
payload = {
    "source": {
        "repo_sha": snapshot.repo_sha,
        "index_sha256": snapshot.file_sha256,
        "dev_end": snapshot.dev_end,
    },
    "experiment": {
        "combination": COMBINATION,
        "return_features": list(RETURN_FEATURES),
        "model": MODEL_NAME,
    },
    "summary": summary,
    "weight_counts": result.weight_counts.to_dict(orient="records"),
}
cache_path = project_root / "data" / "raw" / "model_results" / CACHE_NAME
cache_path.parent.mkdir(parents=True, exist_ok=True)
cache_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("비교용 실행 결과 저장:", cache_path)


,결과
전체 OOS 표본,720.0000
Accuracy,0.3806
Macro F1,0.3659
하락 Recall,0.2615
핵심지표 조화평균,0.3266
Balanced Accuracy,0.3660
최빈 클래스 기준선,0.4250
ΔSharpe_net 폴드 중앙값,0.2808
전 기간 현금 폴드,0.0000


,예측 하락,예측 중립,예측 상승
실제 하락,51,80,64
실제 중립,88,140,78
실제 상승,58,78,83


,precision,recall,f1-score,support
하락,0.2589,0.2615,0.2602,195.0000
중립,0.4698,0.4575,0.4636,306.0000
상승,0.3689,0.3790,0.3739,219.0000
accuracy,0.3806,0.3806,0.3806,0.3806
macro avg,0.3659,0.3660,0.3659,720.0000
weighted avg,0.3820,0.3806,0.3812,720.0000


비교용 실행 결과 저장: C:\Users\Administrator\Alpha_Stack\data\raw\model_results\A_five_day_XGBoost.json


In [4]:
# 05-백테스트·성과 형식으로 핵심 결과를 함께 해석합니다.
display(Markdown(f"""
## 결과 해석

- 전체 OOS Accuracy: `{summary['accuracy']:.4f}`
- 전체 OOS Macro F1: `{summary['macro_f1']:.4f}`
- 하락 Recall: `{summary['down_recall']:.4f}`
- 세 핵심지표 조화평균: `{summary['core_harmonic_mean']:.4f}`
- 최빈 클래스 Accuracy 기준선: `{summary['majority_accuracy']:.4f}`
- 비용 차감 ΔSharpe 폴드 중앙값: `{summary['delta_sharpe_net_median']:.4f}`
- 전 기간 현금 보유 폴드: `{summary['all_cash_folds']}`개
- 예측 개수: 하락 `{summary['predicted_down']}` · 중립
  `{summary['predicted_neutral']}` · 상승 `{summary['predicted_up']}`

이 값은 봉인 홀드아웃이 아닌 개발구간의 12폴드 OOS 결과다. 클래스 가중치는 각 폴드의
외부 검증값을 보지 않고, 그보다 앞선 내부 60거래일의 조화평균으로만 선택했다.
"""))



## 결과 해석

- 전체 OOS Accuracy: `0.3806`
- 전체 OOS Macro F1: `0.3659`
- 하락 Recall: `0.2615`
- 세 핵심지표 조화평균: `0.3266`
- 최빈 클래스 Accuracy 기준선: `0.4250`
- 비용 차감 ΔSharpe 폴드 중앙값: `0.2808`
- 전 기간 현금 보유 폴드: `0`개
- 예측 개수: 하락 `197` · 중립
  `298` · 상승 `225`

이 값은 봉인 홀드아웃이 아닌 개발구간의 12폴드 OOS 결과다. 클래스 가중치는 각 폴드의
외부 검증값을 보지 않고, 그보다 앞선 내부 60거래일의 조화평균으로만 선택했다.
